# K-Turn Hidden State Extraction

Extracts `h_inst` and `h_post_inst` at the Zhao et al. token positions under
controlled context conditions.  See `docs/experiment_design.md` for full details.

### Extraction conditions

| Condition | Input | Granularity | Status |
|-----------|-------|-------------|--------|
| **1 — Full-context** | turns 1..k (prior assistant replies included) | per (conv, k) | Already extracted in nb03 (`data/representations/trajectories/`) |
| **2 — No-context** | system prompt + user_k only (no history) | per (conv, k) | **This notebook** |
| **3 — Compressed** | all turns concatenated into one user message | per conv | **This notebook** |
| **4 — Single-turn baseline** | raw JBB goal, no attack framing | per goal | Already extracted in nb03 (`data/representations/single_turn/`) |

### Token positions

```
... [user content] <|eot_id|>
         ^              ^
      t_inst       t_post_inst
```

### Layers

All 32 transformer layers. Output arrays have shape `(N, 32, 4096)` float16,
matching the nb03 trajectory format.

### Output

- `data/representations/nocontext/{FOLDER_NAME}/` — Condition 2
- `data/representations/compressed/{FOLDER_NAME}/` — Condition 3

In [ ]:
import os
# ── Set visible GPUs before importing torch ───────────────────────────────────
GPU_IDS = [4, 5, 6]                    # <- physical GPU IDs
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
FOLDER_NAME = "crescendo_harmful"      # <- change per run

MODEL_ID  = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE     = torch.bfloat16

# All 32 transformer layers (hidden_states[0] = embedding, skip)
LAYER_INDICES = list(range(1, 33))     # indices into outputs.hidden_states
N_LAYERS      = len(LAYER_INDICES)     # 32

CONV_DIR   = repo_root / "data" / "conversations" / FOLDER_NAME
REPR_ROOT  = repo_root / "data" / "representations"

parts      = FOLDER_NAME.split("_")
FRAMEWORK  = parts[0]
SPLIT      = parts[1]

assert CONV_DIR.exists(), f"Conversation folder not found: {CONV_DIR}"
files = sorted(CONV_DIR.glob("*.json"))

print(f"Folder:    {FOLDER_NAME}")
print(f"Framework: {FRAMEWORK}  Split: {SPLIT}")
print(f"Files:     {len(files)}")
print(f"GPUs:      {GPU_IDS} -> logical {LOGICAL_GPU_IDS}")
print(f"Layers:    {LAYER_INDICES[0]}..{LAYER_INDICES[-1]}  ({N_LAYERS} total)")

In [ ]:
# ── Message builders ──────────────────────────────────────────────────────────

def get_accepted_turns(conv):
    """
    Return list of accepted (non-rolled-back) turn pairs.
    Each element: (user_content, asst_content, asst_turn_dict)

    Crescendo has rolled_back / is_refusal fields; ActorAttack and X-Teaming
    do not (all their turns are accepted).
    """
    turns = conv.get("turns", [])
    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    accepted = []
    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_t = pair.get("user")
        asst_t = pair.get("assistant")
        if not user_t or not asst_t:
            continue
        # Skip rolled-back turns (Crescendo only; other frameworks lack this field)
        if user_t.get("rolled_back", False) or asst_t.get("rolled_back", False):
            continue
        accepted.append((user_t["content"], asst_t["content"], asst_t))
    return accepted


def build_nocontext_messages(conv, k):
    """
    Condition 2: system prompt + user_k message only — no prior history.
    """
    accepted = get_accepted_turns(conv)
    system_prompt = conv.get("target_system_prompt", "")

    if not accepted or k > len(accepted):
        return None

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": accepted[k - 1][0]})
    return messages


def build_compressed_messages(conv):
    """
    Condition 3: all turns concatenated into a single user message.
    Ends after the final user message (no trailing assistant response),
    matching the full-context condition at k = n.
    """
    accepted = get_accepted_turns(conv)
    system_prompt = conv.get("target_system_prompt", "")

    if not accepted:
        return None

    parts = []
    for user_c, asst_c, _ in accepted[:-1]:
        parts.append(f"User: {user_c}\nAssistant: {asst_c}")
    parts.append(f"User: {accepted[-1][0]}")

    compressed = "\n\n".join(parts)

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": compressed})
    return messages


def build_fullcontext_messages(conv, k):
    """
    Condition 1 (for sanity checks only — bulk extraction already in nb03).
    Full-context prefix at turn k: turns 1..k-1 with assistant replies, then user_k.
    """
    accepted = get_accepted_turns(conv)
    system_prompt = conv.get("target_system_prompt", "")

    if not accepted or k > len(accepted):
        return None

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    for user_c, asst_c, _ in accepted[:k - 1]:
        messages.append({"role": "user",      "content": user_c})
        messages.append({"role": "assistant", "content": asst_c})

    messages.append({"role": "user", "content": accepted[k - 1][0]})
    return messages


print("Message builders defined.")

In [ ]:
# ── Token positions & extraction ──────────────────────────────────────────────

def get_positions(tokenizer, input_ids):
    """
    t_post_inst = last <|eot_id|> in the sequence (closes the final user turn)
    t_inst      = t_post_inst - 1  (last content token of the user message)
    """
    eot_id  = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    eot_pos = (input_ids[0] == eot_id).nonzero(as_tuple=True)[0]
    t_post_inst = eot_pos[-1].item()
    t_inst      = t_post_inst - 1
    return t_inst, t_post_inst


@torch.no_grad()
def extract_at_positions(model, tokenizer, messages):
    """
    Single forward pass.  Returns (h_inst, h_post_inst) as (32, 4096) float16
    numpy arrays, or (None, None, ...) on OOM.
    """
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=False
    ).to(model.device)

    t_inst, t_post_inst = get_positions(tokenizer, input_ids)
    seq_len = input_ids.shape[1]

    try:
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            outputs = model(input_ids, output_hidden_states=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None, t_inst, t_post_inst, seq_len

    h_inst = np.stack([
        outputs.hidden_states[l][0, t_inst, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])  # (32, 4096)

    h_post = np.stack([
        outputs.hidden_states[l][0, t_post_inst, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])  # (32, 4096)

    del outputs, input_ids
    torch.cuda.empty_cache()

    return h_inst, h_post, t_inst, t_post_inst, seq_len


def save_arrays(save_dir, h_inst_list, h_post_list, meta_list):
    """Concatenate and save arrays + metadata."""
    save_dir.mkdir(parents=True, exist_ok=True)
    h_inst = np.concatenate(h_inst_list, axis=0).astype(np.float16)
    h_post = np.concatenate(h_post_list, axis=0).astype(np.float16)
    meta   = pd.concat(meta_list, ignore_index=True)
    np.save(str(save_dir / "h_inst.npy"), h_inst)
    np.save(str(save_dir / "h_post_inst.npy"), h_post)
    meta.to_parquet(save_dir / "metadata.parquet", index=False)
    print(f"  Saved -> {save_dir}")
    print(f"  h_inst:      {h_inst.shape}  ({h_inst.nbytes / 1e9:.2f} GB)")
    print(f"  h_post_inst: {h_post.shape}  ({h_post.nbytes / 1e9:.2f} GB)")
    print(f"  rows:        {len(meta)}")


print("Extraction helpers defined.")

## Sanity checks

Run on a handful of conversations before launching full extraction.
Checks from `docs/experiment_design.md` sections 1-4 and 7.

In [ ]:
tok_check = AutoTokenizer.from_pretrained(MODEL_ID)

# ── Check 1: Token position verification ─────────────────────────────────────
print("=" * 60)
print("CHECK 1 -- Token positions (add_generation_prompt=False)")
print("=" * 60)

for fpath in files[:5]:
    conv = json.loads(fpath.read_text())
    accepted = get_accepted_turns(conv)
    if not accepted:
        print(f"  {fpath.name}: SKIPPED (no accepted turns)")
        continue

    # Full-context at final turn
    messages = build_fullcontext_messages(conv, len(accepted))
    input_ids = tok_check.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=False
    )
    t_inst, t_post = get_positions(tok_check, input_ids)

    tok_inst = tok_check.decode([input_ids[0][t_inst].item()])
    tok_post = tok_check.decode([input_ids[0][t_post].item()])
    context  = tok_check.decode(input_ids[0][t_inst - 3 : t_post + 2].tolist())

    ok_post = tok_post == "<|eot_id|>"
    ok_inst = bool(tok_inst.strip())

    print(f"  {fpath.name}  turns={len(accepted)}")
    print(f"    t_inst [{t_inst}]      = {repr(tok_inst):20s}  {'OK' if ok_inst else 'WARN: whitespace'}")
    print(f"    t_post_inst [{t_post}] = {repr(tok_post):20s}  {'OK' if ok_post else 'WARN: not eot_id'}")
    print(f"    context: {repr(context)}")
    print()

# Also check no-context and compressed on one conversation
conv0 = json.loads(files[0].read_text())
accepted0 = get_accepted_turns(conv0)

print("-- No-context (k=1) --")
nc_msgs = build_nocontext_messages(conv0, 1)
nc_ids  = tok_check.apply_chat_template(nc_msgs, return_tensors="pt", add_generation_prompt=False)
t_i, t_p = get_positions(tok_check, nc_ids)
print(f"  t_inst={repr(tok_check.decode([nc_ids[0][t_i].item()]))}  "
      f"t_post={repr(tok_check.decode([nc_ids[0][t_p].item()]))}")

print("-- Compressed --")
comp_msgs = build_compressed_messages(conv0)
comp_ids  = tok_check.apply_chat_template(comp_msgs, return_tensors="pt", add_generation_prompt=False)
t_i, t_p = get_positions(tok_check, comp_ids)
print(f"  t_inst={repr(tok_check.decode([comp_ids[0][t_i].item()]))}  "
      f"t_post={repr(tok_check.decode([comp_ids[0][t_p].item()]))}")

In [ ]:
# ── Check 2: Prefix construction ──────────────────────────────────────────────
print("=" * 60)
print("CHECK 2 -- Prefix construction")
print("=" * 60)

conv = json.loads(files[0].read_text())
accepted = get_accepted_turns(conv)

for k in [1, 2, len(accepted)]:
    msgs = build_fullcontext_messages(conv, k)
    # Expected: 1 system + k user + (k-1) assistant = 2k messages
    expected_len = 1 + k + (k - 1)
    actual_len = len(msgs)
    status = "OK" if actual_len == expected_len else f"MISMATCH (expected {expected_len})"
    print(f"  k={k}: {actual_len} messages  {status}")
    for m in msgs:
        print(f"    [{m['role']:9s}] {m['content'][:80]}...")
    print()

# ── Check 3: No-context message content matches full-context ─────────────────
print("=" * 60)
print("CHECK 3 -- No-context user content == full-context user content at each k")
print("=" * 60)

for k in [1, 2, len(accepted)]:
    full_msgs = build_fullcontext_messages(conv, k)
    nc_msgs   = build_nocontext_messages(conv, k)
    match = full_msgs[-1]["content"] == nc_msgs[-1]["content"]
    print(f"  k={k}: full={len(full_msgs)} msgs, nocontext={len(nc_msgs)} msgs  "
          f"user content match: {match}")
    assert match, f"Content mismatch at k={k}!"

# ── Check 4: Rolled-back turn filtering ──────────────────────────────────────
print()
print("=" * 60)
print("CHECK 4 -- Rolled-back turns filtered correctly")
print("=" * 60)

for fpath in files[:10]:
    conv = json.loads(fpath.read_text())
    n_accepted = len(get_accepted_turns(conv))
    expected   = conv.get("executed_turns", n_accepted)
    status = "OK" if n_accepted == expected else f"MISMATCH (executed_turns={expected})"
    print(f"  {fpath.name}: accepted={n_accepted}  {status}")

## Condition 2 — No-context extraction

For each conversation of length n, runs n forward passes where each pass
contains **only** the system prompt + user_k message (no prior history).

The difference between Condition 1 (full-context) and Condition 2 (no-context)
at the same (conv, k) isolates the **pure effect of context accumulation** on
the hidden state. Anything that differs cannot be due to message content.

In [ ]:
SAVE_NC = REPR_ROOT / "nocontext" / FOLDER_NAME
SAVE_NC.mkdir(parents=True, exist_ok=True)

# ── Resume support ────────────────────────────────────────────────────────────
meta_path_nc = SAVE_NC / "metadata.parquet"
if meta_path_nc.exists():
    _existing_meta_nc = pd.read_parquet(meta_path_nc)
    done_nc = set(zip(
        _existing_meta_nc["pair_id"],
        _existing_meta_nc["attempt"],
        _existing_meta_nc["turn_k"],
    ))
    existing_h_inst_nc = [np.load(str(SAVE_NC / "h_inst.npy"))]
    existing_h_post_nc = [np.load(str(SAVE_NC / "h_post_inst.npy"))]
    existing_meta_nc   = [_existing_meta_nc]
    print(f"Resuming: {len(done_nc)} (pair_id, attempt, turn_k) triples already done")
else:
    done_nc = set()
    existing_h_inst_nc = existing_h_post_nc = existing_meta_nc = []

# ── Count total work ──────────────────────────────────────────────────────────
total_pairs = sum(
    len(get_accepted_turns(json.loads(f.read_text())))
    for f in files
)
print(f"Total (conversation, turn) pairs: {total_pairs}")
print(f"Already done: {len(done_nc)}")
print(f"Remaining: {total_pairs - len(done_nc)}")

In [ ]:
n_gpus  = len(LOGICAL_GPU_IDS)
chunks  = [files[i::n_gpus] for i in range(n_gpus)]

all_h_inst_nc = list(existing_h_inst_nc)
all_h_post_nc = list(existing_h_post_nc)
all_meta_nc   = list(existing_meta_nc)

pbar_nc = tqdm(total=total_pairs - len(done_nc), desc="No-context",
               file=sys.stdout, dynamic_ncols=True)
lock_nc = threading.Lock()


def worker_nc(gpu_id, chunk):
    torch.cuda.set_device(gpu_id)
    device = f"cuda:{gpu_id}"
    print(f"GPU {gpu_id}: loading model...", flush=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, device_map={"": device}
    )
    mdl.eval()
    print(f"GPU {gpu_id}: ready, {len(chunk)} files", flush=True)

    batch_h_inst, batch_h_post, batch_meta = [], [], []
    for fpath in chunk:
        conv     = json.loads(fpath.read_text())
        pair_id  = conv["objective_pair_id"]
        attempt  = conv.get("attempt", 1)
        accepted = get_accepted_turns(conv)
        n_accepted = len(accepted)
        if n_accepted == 0:
            continue

        for k in range(1, n_accepted + 1):
            if (pair_id, attempt, k) in done_nc:
                continue

            messages = build_nocontext_messages(conv, k)
            if messages is None:
                with lock_nc:
                    pbar_nc.update(1)
                continue

            h_inst, h_post, t_inst, t_post, seq_len = extract_at_positions(
                mdl, tok, messages
            )
            if h_inst is None:
                print(f"  OOM: {fpath.name} k={k} seq_len={seq_len}", flush=True)
                with lock_nc:
                    pbar_nc.update(1)
                continue

            _, _, asst_turn = accepted[k - 1]
            batch_h_inst.append(h_inst)
            batch_h_post.append(h_post)
            batch_meta.append({
                "conversation_id":  conv.get("conversation_id", fpath.stem),
                "pair_id":          pair_id,
                "goal_type":        conv.get("goal_type", SPLIT),
                "framework":        FRAMEWORK,
                "attempt":          attempt,
                "attack_success":   bool(conv.get("attack_success", False)),
                "n_accepted_turns": n_accepted,
                "turn_k":           k,
                "is_refusal":       bool(asst_turn.get("is_refusal", False)),
                "judge_success":    bool(asst_turn.get("judge_success", False)),
                "seq_len":          seq_len,
                "t_inst":           t_inst,
                "t_post_inst":      t_post,
                "fname":            fpath.name,
            })
            with lock_nc:
                pbar_nc.update(1)

    with lock_nc:
        if batch_h_inst:
            all_h_inst_nc.append(np.stack(batch_h_inst))
            all_h_post_nc.append(np.stack(batch_h_post))
            all_meta_nc.append(pd.DataFrame(batch_meta))


threads = [threading.Thread(target=worker_nc, args=(g, c))
           for g, c in zip(LOGICAL_GPU_IDS, chunks)]
for t in threads:
    t.start()
for t in threads:
    t.join()
pbar_nc.close()

save_arrays(SAVE_NC, all_h_inst_nc, all_h_post_nc, all_meta_nc)

## Condition 3 — Compressed single-turn extraction

The entire multi-turn conversation is concatenated into a **single user message**
with no turn structure (no role headers, no `<|eot_id|>` between turns).
One extraction per conversation.

If compressed representations match full-context, then semantic content alone
drives the representation shift. If they differ, the turn structure and
role-based attention patterns matter independently (Bullwinkel et al. found
they were nearly identical for Crescendo — this is a replication check).

In [ ]:
SAVE_COMP = REPR_ROOT / "compressed" / FOLDER_NAME
SAVE_COMP.mkdir(parents=True, exist_ok=True)

# ── Resume support ────────────────────────────────────────────────────────────
meta_path_comp = SAVE_COMP / "metadata.parquet"
if meta_path_comp.exists():
    _existing_meta_comp = pd.read_parquet(meta_path_comp)
    done_comp = set(zip(
        _existing_meta_comp["pair_id"],
        _existing_meta_comp["attempt"],
    ))
    existing_h_inst_comp = [np.load(str(SAVE_COMP / "h_inst.npy"))]
    existing_h_post_comp = [np.load(str(SAVE_COMP / "h_post_inst.npy"))]
    existing_meta_comp   = [_existing_meta_comp]
    print(f"Resuming: {len(done_comp)} conversations already done")
else:
    done_comp = set()
    existing_h_inst_comp = existing_h_post_comp = existing_meta_comp = []

print(f"Total conversations: {len(files)}")
print(f"Already done: {len(done_comp)}")
print(f"Remaining: {len(files) - len(done_comp)}")

In [ ]:
chunks_comp = [files[i::n_gpus] for i in range(n_gpus)]

all_h_inst_comp = list(existing_h_inst_comp)
all_h_post_comp = list(existing_h_post_comp)
all_meta_comp   = list(existing_meta_comp)

pbar_comp = tqdm(total=len(files) - len(done_comp), desc="Compressed",
                 file=sys.stdout, dynamic_ncols=True)
lock_comp = threading.Lock()


def worker_comp(gpu_id, chunk):
    torch.cuda.set_device(gpu_id)
    device = f"cuda:{gpu_id}"
    print(f"GPU {gpu_id}: loading model...", flush=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, device_map={"": device}
    )
    mdl.eval()
    print(f"GPU {gpu_id}: ready, {len(chunk)} files", flush=True)

    batch_h_inst, batch_h_post, batch_meta = [], [], []
    for fpath in chunk:
        conv     = json.loads(fpath.read_text())
        pair_id  = conv["objective_pair_id"]
        attempt  = conv.get("attempt", 1)

        if (pair_id, attempt) in done_comp:
            continue

        accepted = get_accepted_turns(conv)
        if not accepted:
            with lock_comp:
                pbar_comp.update(1)
            continue

        messages = build_compressed_messages(conv)
        if messages is None:
            with lock_comp:
                pbar_comp.update(1)
            continue

        h_inst, h_post, t_inst, t_post, seq_len = extract_at_positions(
            mdl, tok, messages
        )
        if h_inst is None:
            print(f"  OOM: {fpath.name} seq_len={seq_len}", flush=True)
            with lock_comp:
                pbar_comp.update(1)
            continue

        batch_h_inst.append(h_inst)
        batch_h_post.append(h_post)
        batch_meta.append({
            "conversation_id":  conv.get("conversation_id", fpath.stem),
            "pair_id":          pair_id,
            "goal_type":        conv.get("goal_type", SPLIT),
            "framework":        FRAMEWORK,
            "attempt":          attempt,
            "attack_success":   bool(conv.get("attack_success", False)),
            "n_accepted_turns": len(accepted),
            "seq_len":          seq_len,
            "t_inst":           t_inst,
            "t_post_inst":      t_post,
            "fname":            fpath.name,
        })
        with lock_comp:
            pbar_comp.update(1)

    with lock_comp:
        if batch_h_inst:
            all_h_inst_comp.append(np.stack(batch_h_inst))
            all_h_post_comp.append(np.stack(batch_h_post))
            all_meta_comp.append(pd.DataFrame(batch_meta))


threads_comp = [threading.Thread(target=worker_comp, args=(g, c))
                for g, c in zip(LOGICAL_GPU_IDS, chunks_comp)]
for t in threads_comp:
    t.start()
for t in threads_comp:
    t.join()
pbar_comp.close()

save_arrays(SAVE_COMP, all_h_inst_comp, all_h_post_comp, all_meta_comp)

## Cross-condition verification

**Check 5 — No-context k=1 vs trajectory k=1:** At k=1 there is no prior context, so both
conditions receive the identical input (system + user_1). Cosine similarity should be ~1.0.

**Check 6 — Divergence at k>1:** Full-context and no-context should diverge as k grows,
confirming that context accumulation is actually changing the hidden states.

Both checks compare against the existing trajectory data from nb03.
Both arrays are `(N, 32, 4096)` with the same layer ordering.

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
traj_dir = REPR_ROOT / "trajectories" / FOLDER_NAME

if not traj_dir.exists():
    print(f"Trajectory data not found at {traj_dir} -- skipping cross-checks.")
else:
    traj_meta = pd.read_parquet(traj_dir / "metadata.parquet")
    traj_h    = np.load(str(traj_dir / "h_inst.npy"), mmap_mode="r")

    nc_meta = pd.read_parquet(SAVE_NC / "metadata.parquet")
    nc_h    = np.load(str(SAVE_NC / "h_inst.npy"), mmap_mode="r")

    CHECK_LAYERS = [13, 19, 31]   # L14, L20, L32 (0-indexed)
    CHECK_LABELS = ["L14", "L20", "L32"]

    def cosine(a, b):
        a, b = a.astype(np.float32), b.astype(np.float32)
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

    # ── Check 5: k=1, both conditions identical input ─────────────────────────
    print("=" * 60)
    print("CHECK 5 -- No-context k=1 vs trajectory k=1 (should be ~1.0)")
    print("=" * 60)

    traj_k1 = traj_meta[traj_meta["turn_k"] == 1].head(10)
    for _, row in traj_k1.iterrows():
        nc_match = nc_meta[
            (nc_meta["pair_id"] == row["pair_id"])
            & (nc_meta["attempt"] == row["attempt"])
            & (nc_meta["turn_k"] == 1)
        ]
        if nc_match.empty:
            continue

        traj_idx = row.name
        nc_idx   = nc_match.index[0]

        sims = []
        for li, ll in zip(CHECK_LAYERS, CHECK_LABELS):
            sims.append(cosine(traj_h[traj_idx, li, :], nc_h[nc_idx, li, :]))

        print(f"  pair={row['pair_id']:3d} att={row['attempt']}  "
              + "  ".join(f"{ll}={s:.6f}" for ll, s in zip(CHECK_LABELS, sims)))

    # ── Check 6: divergence at k>1 ───────────────────────────────────────────
    print()
    print("=" * 60)
    print("CHECK 6 -- No-context vs trajectory diverge at k>1")
    print("=" * 60)

    for k_check in [1, 3, 5, 8]:
        traj_rows = traj_meta[traj_meta["turn_k"] == k_check].head(30)
        sims = []
        for _, row in traj_rows.iterrows():
            nc_match = nc_meta[
                (nc_meta["pair_id"] == row["pair_id"])
                & (nc_meta["attempt"] == row["attempt"])
                & (nc_meta["turn_k"] == k_check)
            ]
            if nc_match.empty:
                continue
            # Compare at L20 (index 19)
            sims.append(cosine(traj_h[row.name, 19, :], nc_h[nc_match.index[0], 19, :]))
        if sims:
            print(f"  k={k_check:2d}: mean cos(traj, nocontext) L20 = "
                  f"{np.mean(sims):.4f} +/- {np.std(sims):.4f}  (n={len(sims)})")

    print()
    print("Expected: k=1 ~ 1.0; k>1 progressively < 1.0")